# Tutorial de Seaborn para Visualización de Datos
Basado en el tutorial de **coursera's IBM Python for Data Science and AI y notas del Dr. Ivan Olier-Caparroso**

## Introducción
**Seaborn** es una librería de visualización de datos en Python basada en Matplotlib. Está diseñada para crear gráficos estadísticos atractivos con muy pocas líneas de código. Sus principales ventajas son:

- Integración nativa con DataFrames de Pandas
- Paletas de colores atractivas por defecto
- Funciones de alto nivel para gráficos estadísticos complejos
- Fácil personalización

En este notebook aprenderás a crear y personalizar los tipos de gráficos más utilizados en Ciencia de Datos.

<hr/>
<div class="alert alert-success alertsuccess" style="margin-top: 20px">
[Consejo]: Para ejecutar el código en una celda, haz clic en ella y presiona <kbd>Shift</kbd> + <kbd>Enter</kbd>.
</div>
<hr/>

In [ ]:
# Importar librerías necesarias
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Configuración global de estilo
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 5)

print('Versión de Seaborn:', sns.__version__)

## 1. Datasets de Ejemplo

Seaborn incluye varios datasets de ejemplo que podemos cargar directamente. También crearemos un dataset personalizado:

In [ ]:
# Cargar datasets clásicos de Seaborn
tips    = sns.load_dataset('tips')        # Propinas en un restaurante
iris    = sns.load_dataset('iris')        # Medidas de flores
titanic = sns.load_dataset('titanic')     # Pasajeros del Titanic

print('Dataset TIPS (propinas):')
print(tips.head())
print('\nForma:', tips.shape)

In [ ]:
# Dataset personalizado: rendimiento académico

# np.random.seed(n): fija la semilla del generador de números aleatorios.
# Usando la misma semilla siempre obtendremos los mismos datos 'aleatorios',
# lo que hace que los resultados sean reproducibles entre ejecuciones.
np.random.seed(2024)
n = 200  # número de alumnos simulados

df_alumnos = pd.DataFrame({

    # np.random.normal(loc, scale, size)
    # Genera 'size' números de una distribución normal (campana de Gauss).
    # loc   = media (valor central de la distribución)
    # scale = desviación estándar (qué tan dispersos son los valores)
    # .clip(min, max) recorta los valores fuera del rango realista [0, 12]
    'horas_estudio':  np.random.normal(5, 2, n).clip(0, 12),

    # Misma función: media=70, desv.std=15. Se recorta entre 0 y 100.
    # Esta columna será sobreescrita más abajo para añadir correlación real.
    'calificacion':   np.random.normal(70, 15, n).clip(0, 100),

    # np.random.choice(a, size)
    # Selecciona aleatoriamente 'size' elementos de la lista 'a',
    # con reemplazo (un valor puede repetirse). Útil para variables categóricas.
    'genero':         np.random.choice(['Femenino', 'Masculino'], n),
    'carrera':        np.random.choice(['Ingeniería', 'Medicina', 'Derecho', 'Arte'], n),

    # np.random.randint(low, high, size)
    # Genera 'size' enteros aleatorios en el intervalo [low, high).
    # Nota: el límite superior 'high' es EXCLUIDO, por eso usamos 11 para llegar al semestre 10.
    'semestre':       np.random.randint(1, 11, n),

    # np.random.uniform(low, high, size)
    # Genera 'size' números de una distribución uniforme: todos los valores
    # entre 'low' y 'high' tienen exactamente la misma probabilidad de aparecer.
    # A diferencia de normal(), no hay valores más probables que otros.
    'asistencia':     np.random.uniform(60, 100, n)
})

# Sobreescribimos 'calificacion' con una fórmula que introduce correlación real:
# calificacion = 50 + (horas * 3.5) + ruido_aleatorio
# El ruido np.random.normal(0, 10, n) simula factores externos (nervios, dificultad del examen, etc.)
df_alumnos['calificacion'] = (
    50 + df_alumnos['horas_estudio'] * 3.5 + np.random.normal(0, 10, n)
).clip(0, 100)

df_alumnos.head()

## 2. Gráficos de Distribución

Los gráficos de distribución nos permiten entender cómo se distribuyen los datos en una variable.

### 2.1 Histograma con KDE (histplot)

In [ ]:
# Histograma básico con curva de densidad
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_alumnos['calificacion'], kde=True, bins=20, color='steelblue', ax=axes[0])
axes[0].set_title('Distribución de Calificaciones')
axes[0].set_xlabel('Calificación')
axes[0].set_ylabel('Frecuencia')

# Histograma separado por grupo
sns.histplot(data=df_alumnos, x='calificacion', hue='genero',
             kde=True, bins=20, alpha=0.6, ax=axes[1])
axes[1].set_title('Distribución por Género')
axes[1].set_xlabel('Calificación')

plt.tight_layout()
plt.show()

### 2.2 Gráfico de densidad (kdeplot)

In [ ]:
# KDE por carrera
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df_alumnos, x='calificacion', hue='carrera',
            fill=True, alpha=0.3, linewidth=2)
plt.title('Densidad de Calificaciones por Carrera')
plt.xlabel('Calificación')
plt.ylabel('Densidad')
plt.show()

## 3. Gráficos de Relación

### 3.1 Diagrama de dispersión (scatterplot)

In [ ]:
# Scatterplot: horas de estudio vs calificación
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_alumnos,
    x='horas_estudio',
    y='calificacion',
    hue='genero',
    style='genero',
    alpha=0.7,
    s=60
)
plt.title('Horas de Estudio vs Calificación')
plt.xlabel('Horas de Estudio por Día')
plt.ylabel('Calificación')
plt.show()

### 3.2 Gráfico de regresión (regplot / lmplot)

Seaborn puede ajustar y graficar una línea de regresión automáticamente:

In [ ]:
# Regresión lineal con intervalo de confianza
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_alumnos,
    x='horas_estudio',
    y='calificacion',
    scatter_kws={'alpha': 0.4, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 2}
)
plt.title('Regresión: Horas de Estudio → Calificación')
plt.xlabel('Horas de Estudio por Día')
plt.ylabel('Calificación')
plt.show()

In [ ]:
# lmplot: regresión separada por grupos
g = sns.lmplot(
    data=df_alumnos,
    x='horas_estudio',
    y='calificacion',
    hue='genero',
    height=5,
    aspect=1.8,
    scatter_kws={'alpha': 0.4}
)
g.figure.suptitle('Regresión por Género', y=1.02)
plt.show()

## 4. Gráficos Categóricos

### 4.1 Diagrama de caja (boxplot)

El boxplot muestra la distribución de una variable numérica para diferentes categorías, incluyendo mediana, cuartiles y valores atípicos:

In [ ]:
# Boxplot de calificaciones por carrera
plt.figure(figsize=(11, 6))
sns.boxplot(
    data=df_alumnos,
    x='carrera',
    y='calificacion',
    hue='genero',
    palette='Set2'
)
plt.title('Distribución de Calificaciones por Carrera y Género')
plt.xlabel('Carrera')
plt.ylabel('Calificación')
plt.legend(title='Género')
plt.show()

### 4.2 Diagrama de violín (violinplot)

Combina el boxplot con un KDE para mostrar la forma completa de la distribución:

In [ ]:
# Violin plot
plt.figure(figsize=(11, 6))
sns.violinplot(
    data=df_alumnos,
    x='carrera',
    y='calificacion',
    hue='genero',
    split=True,
    palette='Pastel1'
)
plt.title('Distribución de Calificaciones (Violin Plot)')
plt.xlabel('Carrera')
plt.ylabel('Calificación')
plt.show()

### 4.3 Gráfico de barras (barplot)

In [ ]:
# Barplot: calificación promedio por carrera
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_alumnos,
    x='carrera',
    y='calificacion',
    palette='Blues_d',
    capsize=0.1
)
plt.title('Calificación Promedio por Carrera (con IC 95%)')
plt.xlabel('Carrera')
plt.ylabel('Calificación Promedio')
plt.show()

### 4.4 Gráfico de conteo (countplot)

In [ ]:
# Countplot: número de alumnos por carrera y género
plt.figure(figsize=(10, 5))
sns.countplot(
    data=df_alumnos,
    x='carrera',
    hue='genero',
    palette='Set1'
)
plt.title('Número de Alumnos por Carrera y Género')
plt.xlabel('Carrera')
plt.ylabel('Cantidad')
plt.legend(title='Género')
plt.show()

## 5. Mapa de Calor (Heatmap)

El heatmap es ideal para visualizar matrices de correlación y datos bidimensionales:

In [ ]:
# Matriz de correlación
columnas_numericas = ['horas_estudio', 'calificacion', 'semestre', 'asistencia']
correlacion = df_alumnos[columnas_numericas].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlacion,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True
)
plt.title('Matriz de Correlación - Rendimiento Académico')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap con el dataset Iris
plt.figure(figsize=(8, 5))
sns.heatmap(
    iris.drop('species', axis=1).corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    square=True
)
plt.title('Correlación en el Dataset Iris')
plt.tight_layout()
plt.show()

## 6. Pairplot (Matriz de Dispersión)

El `pairplot` es uno de los gráficos más útiles para la exploración inicial de datos, ya que muestra todas las combinaciones de pares de variables:

In [ ]:
# Pairplot del dataset Iris
g = sns.pairplot(
    iris,
    hue='species',
    diag_kind='kde',
    plot_kws={'alpha': 0.6},
    palette='Set2'
)
g.figure.suptitle('Pairplot - Dataset Iris', y=1.02, fontsize=14)
plt.show()

## 7. FacetGrid y Subplots con Seaborn

Seaborn permite crear múltiples gráficos en una cuadrícula automáticamente:

In [ ]:
# FacetGrid: un gráfico por carrera
g = sns.FacetGrid(df_alumnos, col='carrera', hue='genero', col_wrap=2, height=4)
g.map(sns.scatterplot, 'horas_estudio', 'calificacion', alpha=0.5)
g.add_legend(title='Género')
g.set_axis_labels('Horas de Estudio', 'Calificación')
g.figure.suptitle('Relación Estudio-Calificación por Carrera', y=1.02)
plt.show()

## 8. Aplicación con el Dataset de Tips (Propinas)

Realizamos un análisis visual completo sobre el dataset de propinas de restaurante:

In [ ]:
# Vista general del dataset tips
print(tips.info())
tips.head()

In [ ]:
# Dashboard de 4 gráficos sobre propinas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis de Propinas en Restaurante', fontsize=16, fontweight='bold')

# 1. Distribución de propinas
sns.histplot(tips['tip'], kde=True, ax=axes[0, 0], color='steelblue', bins=20)
axes[0, 0].set_title('Distribución de Propinas')
axes[0, 0].set_xlabel('Propina (USD)')

# 2. Propina vs cuenta total por día
sns.scatterplot(data=tips, x='total_bill', y='tip',
                hue='day', style='time', alpha=0.7, ax=axes[0, 1])
axes[0, 1].set_title('Propina vs Cuenta Total')

# 3. Boxplot por día de la semana
sns.boxplot(data=tips, x='day', y='tip', palette='Set3', ax=axes[1, 0])
axes[1, 0].set_title('Propinas por Día de la Semana')

# 4. Violin plot por género y hora
sns.violinplot(data=tips, x='time', y='tip', hue='sex',
               split=True, palette='Pastel1', ax=axes[1, 1])
axes[1, 1].set_title('Propinas: Almuerzo vs Cena por Género')

plt.tight_layout()
plt.show()

## 9. Personalización de Gráficos

Seaborn ofrece múltiples estilos y paletas de colores para personalizar la apariencia:

In [ ]:
# Comparar diferentes estilos de Seaborn
estilos = ['darkgrid', 'whitegrid', 'dark', 'white', 'ticks']

fig, axes = plt.subplots(1, 5, figsize=(18, 3))

for ax, estilo in zip(axes, estilos):
    with sns.axes_style(estilo):
        datos_demo = np.random.normal(0, 1, 100)
        sns.histplot(datos_demo, kde=True, ax=ax, color='coral')
        ax.set_title(f'Estilo:\n{estilo}', fontsize=10)
        ax.set_xlabel('')

plt.suptitle('Comparación de Estilos en Seaborn', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Paletas de colores disponibles
# Nota: sns.palplot() no acepta 'ax', usamos matplotlib directamente
paletas = ['deep', 'muted', 'bright', 'pastel', 'dark', 'colorblind']

fig, axes = plt.subplots(len(paletas), 1, figsize=(10, len(paletas) * 0.7))

for ax, paleta in zip(axes, paletas):
    colores = sns.color_palette(paleta)
    for i, color in enumerate(colores):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
    ax.set_xlim(0, len(colores))
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylabel(paleta, rotation=0, labelpad=65, va='center', fontsize=10)

plt.suptitle('Paletas de Colores en Seaborn', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Ejercicio Final

Usando el dataset `titanic` de Seaborn, crea un análisis visual que responda las siguientes preguntas:

1. ¿Cuál fue la tasa de supervivencia por clase de pasajero (`pclass`)?
2. ¿Cómo se distribuyen las edades de los supervivientes vs los no supervivientes?
3. ¿Hay diferencia en la tasa de supervivencia entre hombres y mujeres?

Crea un dashboard de 3 gráficos usando subplots.

In [ ]:
# Escribe tu solución aquí
# Primero explora el dataset
print(titanic.columns.tolist())
titanic.head()


Haz doble clic **aquí** para ver una posible solución.

<!--
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Análisis de Supervivencia en el Titanic', fontsize=14)

# 1. Tasa de supervivencia por clase
sns.barplot(data=titanic, x='pclass', y='survived', palette='Blues', ax=axes[0])
axes[0].set_title('Supervivencia por Clase')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Tasa de Supervivencia')

# 2. Distribución de edades (sobrevivientes vs no)
sns.kdeplot(data=titanic, x='age', hue='survived', fill=True, alpha=0.4, ax=axes[1])
axes[1].set_title('Edad: Supervivientes vs No Supervivientes')
axes[1].set_xlabel('Edad')

# 3. Supervivencia por género
sns.barplot(data=titanic, x='sex', y='survived', palette='Set1', ax=axes[2])
axes[2].set_title('Supervivencia por Género')
axes[2].set_xlabel('Género')
axes[2].set_ylabel('Tasa de Supervivencia')

plt.tight_layout()
plt.show()
-->